In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("/home/irene/projects/hackathon-synth-data-2026/data/prepped/mimic-omop-demo-mortality/mortality_full_with_split.csv")

In [4]:
concept_maps = {
    "gender_concept_id": {
        8507: "Male",
        8532: "Female",
        8551: "Unknown",
        8570: "Ambiguous",
    },
    "race_concept_id": {
        8527: "White",
        8516: "Black or African American",
        8515: "Asian",
        8657: "American Indian or Alaska Native",
        8557: "Native Hawaiian or Other Pacific Islander",
        0: "No matching concept",
    },
    "ethnicity_concept_id": {
        38003563: "Hispanic or Latino",
        38003564: "Not Hispanic or Latino",
        0: "No matching concept",
    },
    "visit_concept_id": {
        9201: "Inpatient Visit",
        262: "Emergency Room and Inpatient Visit",
        8870: "Emergency Room - Hospital Admission",
    },
}

person = pd.read_csv("/home/irene/projects/hackathon-synth-data-2026/data/mimic-demo-omop/mimic-iv-demo-data-in-the-omop-common-data-model-0.9/1_omop_data_csv/person.csv")

print(
    person.loc[
        person["race_concept_id"].isin([
            2000001401, 2000001402, 2000001405
        ]),
        ["race_concept_id", "race_source_value"]
    ].drop_duplicates().sort_values("race_concept_id")
)

race_lookup = (
    person[["race_concept_id", "race_source_value"]]
    .dropna()
    .drop_duplicates()
)

# Verify each ID has only one source label
assert (
    race_lookup.groupby("race_concept_id")
    ["race_source_value"].nunique().max() == 1
)

race_map = (
    race_lookup.drop_duplicates("race_concept_id")
    .set_index("race_concept_id")["race_source_value"]
    .to_dict()
)

# Add to your existing race concept mapping
concept_maps["race_concept_id"].update(race_map)



    race_concept_id race_source_value
43       2000001401           UNKNOWN
52       2000001402             OTHER
97       2000001405  UNABLE TO OBTAIN


In [5]:
for col, mapping in concept_maps.items():
    new_col = col.replace("_concept_id", "")

    values = pd.to_numeric(df[col], errors="coerce")
    unmapped = values.notna() & ~values.isin(mapping)

    if unmapped.any():
        raise ValueError(
            f"Unmapped values in {col}: "
            f"{values[unmapped].unique().tolist()}"
        )

    df[new_col] = values.map(mapping).fillna("Unknown / missing")
    df.drop(columns=col, inplace=True)

# df.to_csv("mortality_with_concept_names.csv", index=False)

In [6]:
df["hospital_mortality"] = df["hospital_mortality"].map({
    0: "control",
    1: "case"
})

In [18]:
df.drop(columns='person_id', inplace=True)

In [19]:

df.to_parquet("/home/irene/projects/hackathon-synth-data-2026/data/prepped/mimic-omop-demo-mortality/for_synthetic-data-generator/mortality_full_with_split.parquet", index=False)